In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import locale

# Intentar establecer el idioma a español para las fechas
try:
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except:
    try:
        locale.setlocale(locale.LC_TIME, 'spanish')
    except:
        pass 

# 1. Configuración de Identidad Visual
PGA_COLORS = {
    'pumpkin': '#ED7D31',
    'yellow': '#F9C035',
    'gray': '#595959',
    'platinum': '#D3D4D9',
    'white': '#FFFFFF'
}

# Inyección de CSS nativo de Streamlit
CSS = f"""
<style>
    .pga-header {{
        font-family: 'Lato', sans-serif;
        color: {PGA_COLORS['gray']};
        border-left: 5px solid {PGA_COLORS['pumpkin']};
        padding-left: 15px;
        margin-top: 30px;
        margin-bottom: 15px;
    }}
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)

LINE_COLORS = {
   'Inflacion BCV': '#D3D4D9',
    'Inf. Acum BCV': '#ED7D31',
    'Inflación Acumulada': '#ED7D31',
    'Tasa bcv': '#F9C035',
    'Deval. Acum BCV': '#595959',
    'Devaluación Acumulada BCV': '#F9C035',
    'Devaluación Acumulada USDT': '#595959'
}

# 2. Carga y preparación de datos optimizada con caché
@st.cache_data
def cargar_datos():
    file_path = 'Indicadores_abril_2026.xlsx'
    df = pd.read_excel(file_path, sheet_name='Data')
    df.columns = df.columns.str.strip()

    df_2025 = df[df['Año'] >= 2025].copy()
    df_2025['Año'] = df_2025['Año'].astype(int)

    meses_dict = {
        'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6,
        'JULIO': 7, 'AGOSTO': 8, 'SEPTIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12
    }

    df_2025['Mes_Num'] = df_2025['Mes'].str.strip().str.upper().map(meses_dict)
    df_2025['Fecha_DT'] = pd.to_datetime(
        df_2025['Año'].astype(str) + '-' + 
        df_2025['Mes_Num'].astype(int).astype(str).str.zfill(2) + '-01'
    )
    df_2025 = df_2025.sort_values('Fecha_DT')
    df_2025['Año'] = df_2025['Año'].astype(str)
    return df_2025

df_2025 = cargar_datos()

# --- FUNCIÓN DE GRÁFICO MAESTRA ---
def plot_pga_master(data, columns, title):
    if not columns: return
    fig, ax = plt.subplots(figsize=(11, 4))
    for col in columns:
        color = LINE_COLORS.get(col, '#D3D4D9')
        ax.plot(data['Fecha_DT'], data[col], marker='o', label=col, linewidth=2.5, color=color)
    
    ax.set_title(title, fontsize=12, color=PGA_COLORS['gray'], fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=45, color=PGA_COLORS['gray'])
    
    if any(c in col for c in ['Inflación', 'Devaluación', 'Acum', 'Acumulada', 'Inflacion']):
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.1%}'))
    else:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'Bs. {x:,.2f}'))
    
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.legend(frameon=False, loc='upper left')
    plt.tight_layout()
    st.pyplot(fig) # Modificado para entorno Streamlit

# --- CONFIGURACIÓN DE NOMBRES ---
opciones_h = {
    'Inflacion BCV': 'Inflación Mes',
    'Inf. Acum BCV': 'Inf. Acum BCV',
    'Deval. Acum BCV': 'Deval. Acum BCV',
    'Tasa bcv': 'Última Tasa BCV'
}

opciones_s = {
    'Inflación Acumulada': 'Inflación',
    'Devaluación Acumulada BCV': 'Deval. BCV',
    'Devaluación Acumulada USDT': 'Deval. USDT'
}

# --- SECCIÓN 1: Histórico ---
st.markdown("<div class='pga-header'><h2>1. Histórico (Métricas Acum)</h2></div>", unsafe_allow_html=True)
sel_h_labels = st.multiselect('Seleccione indicadores:', options=list(opciones_h.values()), default=[])
# Mapeo inverso de etiquetas a columnas del DataFrame
sel_h = [k for k, v in opciones_h.items() if v in sel_h_labels]

if sel_h:
    temp_df = df_2025.copy()
    temp_df['Inf. Acum BCV'] = temp_df['Inflación acumulada BCV']
    temp_df['Deval. Acum BCV'] = temp_df['Devaluación acumulada BCV']
    
    fmt = {c: ('{:.2%}' if 'Tasa' not in c else 'Bs. {:,.2f}') for c in sel_h}
    st.dataframe(temp_df[['Año', 'Mes'] + sel_h].style.format(fmt), hide_index=True)
    plot_pga_master(temp_df, sel_h, "Histórico de Indicadores (Métrica Acum)")

# --- SECCIÓN 2: Simulador ---
st.markdown("<hr style='margin:40px 0;'><div class='pga-header'><h2>2. Simulador (Métricas Acumulada)</h2></div>", unsafe_allow_html=True)

col1, col2 = st.columns([1, 2])
with col1:
    meses_unicos = df_2025['Fecha_DT'].dt.strftime('%Y-%m').unique()
    m_base = st.selectbox('📅 Mes Base:', options=meses_unicos)
with col2:
    sel_s_labels = st.multiselect('Seleccione indicadores a simular:', options=list(opciones_s.values()), default=[])
    sel_s = [k for k, v in opciones_s.items() if v in sel_s_labels]

if sel_s and m_base:
    f_base = pd.to_datetime(m_base)
    df_d = df_2025[df_2025['Fecha_DT'] >= f_base].copy()
    prev = df_2025[df_2025['Fecha_DT'] < f_base].tail(1)
    
    df_d['Inflación Acumulada'] = (1 + df_d['Inflacion BCV']).cumprod() - 1
    
    t_ref = prev['Tasa bcv'].values[0] if not prev.empty else df_d['Tasa bcv'].iloc[0]/(1 + df_d['Inflacion BCV'].iloc[0])
    df_d['Devaluación Acumulada BCV'] = (df_d['Tasa bcv'] / t_ref) - 1
    
    if 'Tasa Promedio USDT' in df_2025.columns:
        tu_ref = prev['Tasa Promedio USDT'].values[0] if not prev.empty else df_d['Tasa Promedio USDT'].iloc[0] / 1.02
        df_d['Devaluación Acumulada USDT'] = (df_d['Tasa Promedio USDT'] / tu_ref) - 1
    else:
        df_d['Devaluación Acumulada USDT'] = 0

    st.dataframe(df_d[['Año', 'Mes'] + sel_s].style.format({c: '{:.2%}' for c in sel_s}), hide_index=True)
    plot_pga_master(df_d, sel_s, f"Simulador: Comportamiento Acumulada desde {m_base}")

# --- SECCIÓN 3: Consulta Promedio ---
st.markdown("<hr style='margin:40px 0;'><div class='pga-header'><h2>3. Consulta de Tasas del Mes (Promedio vs Cierre)</h2></div>", unsafe_allow_html=True)

anos_disponibles = sorted(df_2025['Año'].unique(), reverse=True)
meses_disponibles = list(df_2025['Mes'].unique())

col_ano, col_mes = st.columns(2)
with col_ano:
    ano_sel = st.selectbox('📅 Año:', options=anos_disponibles)
with col_mes:
    mes_sel = st.selectbox('📆 Mes:', options=meses_disponibles)

registro = df_2025[(df_2025['Año'] == ano_sel) & (df_2025['Mes'] == mes_sel)]

if not registro.empty:
    tasa_promedio = registro['Tasa Promedio BCV'].values[0]
    tasa_ultima = registro['Tasa bcv'].values[0]
    
    if pd.isna(tasa_promedio):
        st.warning(f"No hay registro de Tasa Promedio cargado para {mes_sel} {ano_sel}.")
    else:
        html_cards = f"""
        <div style='display: flex; gap: 20px; margin-top: 15px;'>
            <div style='background-color: #f8f9fa; border-left: 5px solid {PGA_COLORS.get('pumpkin', '#F26522')}; padding: 15px; border-radius: 4px; min-width: 250px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                <span style='color: {PGA_COLORS.get('gray', '#58595B')}; font-size: 11px; font-weight: bold; display: block; letter-spacing: 0.5px;'>TASA PROMEDIO BCV (Columna T)</span>
                <span style='font-size: 26px; font-weight: bold; color: #212529; display: block; margin-top: 5px;'>Bs. {tasa_promedio:,.4f}</span>
            </div>
            <div style='background-color: #f8f9fa; border-left: 5px solid #D3D4D9; padding: 15px; border-radius: 4px; min-width: 250px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
                <span style='color: {PGA_COLORS.get('gray', '#58595B')}; font-size: 11px; font-weight: bold; display: block; letter-spacing: 0.5px;'>TASA CIERRE BCV (Columna L)</span>
                <span style='font-size: 26px; font-weight: bold; color: #6c757d; display: block; margin-top: 5px;'>Bs. {tasa_ultima:,.4f}</span>
            </div>
        </div>
        """
        st.markdown(html_cards, unsafe_allow_html=True)
else:
    st.error(f"No se encontraron registros coincidentes para {mes_sel} - {ano_sel}.")